In [1]:
# =============================================================================
# USER SETTINGS
# Edit only this cell
# =============================================================================

# Sample metadata template to complete.
# Place this file inside:
#
# source_metadata/
#
SAMPLE_METADATA_NAME = "sample_metadata_to_fill.csv"


# Reference metadata containing the information to append.
# Place this file inside:
#
# source_metadata/
#
REFERENCE_METADATA_NAME = "reference_metadata.csv"


In [2]:
# =============================================================================
# MetaDiv sample_metadata_assignation
# =============================================================================
#
# Stand-alone metadata assignment utility.
#
# Expected structure:
#
# MetaDiv_metadata/
# ├── MetaDiv_metadata.ipynb
# ├── source_metadata/
# │   ├── sample_metadata_to_fill.csv
# │   └── reference_metadata.csv
# │
# └── merged_metadata/              <- created automatically
#     ├── sample_metadata.csv
#     ├── unmatched_samples.csv
#     └── duplicated_reference_IDs.csv   <- only when needed
#
#
# Matching behavior
# -----------------
#
# - The FIRST column of each input table is used as the matching key.
# - Column names do not need to be identical.
# - Leading and trailing spaces are ignored during matching.
# - Original sample IDs are preserved.
# - A LEFT JOIN is used internally.
# - Only successfully matched samples are written to sample_metadata.csv.
# - Unmatched samples are written to unmatched_samples.csv.
# - Duplicated IDs in the reference table are allowed, but only the FIRST
#   occurrence is used for metadata assignment.
# - If an incoming reference column already exists in the sample table,
#   "_reference" is appended to the incoming column name.
#
# =============================================================================


from pathlib import Path
import pandas as pd


# =============================================================================
# 1. DETECT STAND-ALONE PROJECT DIRECTORY
# =============================================================================
#
# The script searches the current working directory and its parents until it
# finds a folder named:
#
#     source_metadata/
#
# The directory containing source_metadata/ is treated as the project folder.
# =============================================================================


def detect_project_dir():

    current_dir = Path.cwd().resolve()

    candidate_dirs = [
        current_dir,
        *current_dir.parents
    ]

    for candidate in candidate_dirs:

        if (
            candidate
            / "source_metadata"
        ).is_dir():

            return candidate


    raise FileNotFoundError(
        "\nCould not detect the MetaDiv_metadata project directory.\n\n"
        "Create a folder named 'source_metadata' next to the notebook:\n\n"
        "MetaDiv_metadata/\n"
        "├── MetaDiv_metadata.ipynb\n"
        "└── source_metadata/\n"
        "    ├── sample_metadata_to_fill.csv\n"
        "    └── reference_metadata.csv\n"
    )


PROJECT_DIR = detect_project_dir()


# =============================================================================
# 2. INPUT AND OUTPUT DIRECTORIES
# =============================================================================


SOURCE_DIR = (
    PROJECT_DIR
    / "source_metadata"
)


OUTPUT_DIR = (
    PROJECT_DIR
    / "merged_metadata"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INPUT FILES
# =============================================================================


SAMPLE_FILE = (
    SOURCE_DIR
    / SAMPLE_METADATA_NAME
)


REFERENCE_FILE = (
    SOURCE_DIR
    / REFERENCE_METADATA_NAME
)


# =============================================================================
# 4. OUTPUT FILES
# =============================================================================


OUTPUT_FILE = (
    OUTPUT_DIR
    / "sample_metadata.csv"
)


LOG_FILE = (
    OUTPUT_DIR
    / "unmatched_samples.csv"
)


DUPLICATE_LOG_FILE = (
    OUTPUT_DIR
    / "duplicated_reference_IDs.csv"
)


# Remove an old duplicate log so it does not persist from a previous run
# when the current reference table has no duplicated IDs.

if DUPLICATE_LOG_FILE.exists():

    DUPLICATE_LOG_FILE.unlink()


# =============================================================================
# 5. VALIDATE INPUT FILES
# =============================================================================


if not SAMPLE_FILE.exists():

    raise FileNotFoundError(
        "\nSample metadata file not found:\n"
        f"{SAMPLE_FILE}\n\n"
        "Check SAMPLE_METADATA_NAME in the USER SETTINGS cell."
    )


if not REFERENCE_FILE.exists():

    raise FileNotFoundError(
        "\nReference metadata file not found:\n"
        f"{REFERENCE_FILE}\n\n"
        "Check REFERENCE_METADATA_NAME in the USER SETTINGS cell."
    )


# =============================================================================
# 6. READ INPUT FILES
# =============================================================================


sample = pd.read_csv(
    SAMPLE_FILE,
    dtype=str,
    encoding="utf-8-sig"
)


reference = pd.read_csv(
    REFERENCE_FILE,
    dtype=str,
    encoding="utf-8-sig"
)


# =============================================================================
# 7. VALIDATE INPUT TABLES
# =============================================================================


if sample.shape[1] == 0:

    raise ValueError(
        "The sample metadata file does not contain any columns."
    )


if reference.shape[1] == 0:

    raise ValueError(
        "The reference metadata file does not contain any columns."
    )


if len(sample) == 0:

    raise ValueError(
        "The sample metadata file does not contain any rows."
    )


if len(reference) == 0:

    raise ValueError(
        "The reference metadata file does not contain any rows."
    )


# =============================================================================
# 8. IDENTIFY MATCHING COLUMNS
# =============================================================================
#
# The FIRST column from each file is automatically used as the matching key.
# Column names do not need to be identical.
# =============================================================================


sample_key = sample.columns[0]

reference_key = reference.columns[0]


print("=" * 60)
print("MetaDiv sample_metadata_assignation")
print("=" * 60)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nSample metadata:")
print(SAMPLE_FILE)

print("\nReference metadata:")
print(REFERENCE_FILE)

print("\nColumns used for sample matching:")

print(
    f"  Sample metadata    -> {sample_key}"
)

print(
    f"  Reference metadata -> {reference_key}"
)


# =============================================================================
# 9. CREATE NORMALIZED MATCHING KEYS
# =============================================================================
#
# Temporary keys are created without changing the original identifiers.
# Only leading and trailing whitespace is removed.
# =============================================================================


sample["_JOIN_KEY"] = (
    sample[
        sample_key
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


reference["_JOIN_KEY"] = (
    reference[
        reference_key
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


# =============================================================================
# 10. CHECK EMPTY MATCHING IDs
# =============================================================================


empty_sample_keys = (
    sample["_JOIN_KEY"]
    == ""
)


if empty_sample_keys.any():

    print(
        "\nWARNING: empty sample IDs detected "
        "in the sample metadata table:"
    )

    print(
        f"  Empty IDs: "
        f"{int(empty_sample_keys.sum()):,}"
    )


empty_reference_keys = (
    reference["_JOIN_KEY"]
    == ""
)


if empty_reference_keys.any():

    print(
        "\nWARNING: empty IDs detected "
        "in the reference metadata:"
    )

    print(
        f"  Empty IDs: "
        f"{int(empty_reference_keys.sum()):,}"
    )


# =============================================================================
# 11. HANDLE DUPLICATED REFERENCE IDs
# =============================================================================
#
# Duplicated reference IDs could generate multiple rows during the merge.
#
# Therefore:
#
# - all duplicated reference records are exported for review;
# - only the FIRST occurrence is used for metadata assignment.
# =============================================================================


duplicate_reference = reference.duplicated(
    subset=[
        "_JOIN_KEY"
    ],
    keep=False
)


if duplicate_reference.any():

    duplicates = reference.loc[
        duplicate_reference
    ].copy()


    print(
        "\nWARNING: duplicated IDs were detected "
        "in the reference metadata."
    )

    print(
        f"Rows involved: "
        f"{int(duplicate_reference.sum()):,}"
    )

    print(
        f"Unique duplicated IDs: "
        f"{duplicates['_JOIN_KEY'].nunique():,}"
    )

    print(
        "Only the FIRST occurrence of each duplicated ID "
        "will be used."
    )


    duplicates.drop(
        columns=[
            "_JOIN_KEY"
        ],
        errors="ignore"
    ).to_csv(
        DUPLICATE_LOG_FILE,
        index=False,
        encoding="utf-8-sig"
    )


    reference = reference.drop_duplicates(
        subset=[
            "_JOIN_KEY"
        ],
        keep="first"
    ).copy()


else:

    print(
        "\nNo duplicated IDs were detected "
        "in the reference metadata."
    )


# =============================================================================
# 12. IDENTIFY METADATA COLUMNS TO APPEND
# =============================================================================
#
# All columns from the reference table are appended EXCEPT:
#
# - its first column, used as the matching identifier;
# - the temporary internal matching key.
# =============================================================================


reference_columns_to_add = [
    column

    for column in reference.columns

    if column not in {
        reference_key,
        "_JOIN_KEY"
    }
]


print(
    f"\nMetadata columns to append: "
    f"{len(reference_columns_to_add):,}"
)


# =============================================================================
# 13. HANDLE DUPLICATED COLUMN NAMES
# =============================================================================


rename_reference = {}


for column in reference_columns_to_add:

    if column in sample.columns:

        rename_reference[
            column
        ] = (
            f"{column}_reference"
        )


if rename_reference:

    print(
        "\nReference columns renamed "
        "to avoid duplicated column names:"
    )

    for old_name, new_name in rename_reference.items():

        print(
            f"  {old_name} -> {new_name}"
        )


# =============================================================================
# 14. PREPARE REFERENCE TABLE FOR JOIN
# =============================================================================


reference_for_join = reference[
    [
        "_JOIN_KEY"
    ]
    +
    reference_columns_to_add
].copy()


reference_for_join = reference_for_join.rename(
    columns=rename_reference
)


# =============================================================================
# 15. PRESERVE ORIGINAL SAMPLE ORDER
# =============================================================================


sample["_ORIGINAL_ORDER"] = range(
    len(sample)
)


# =============================================================================
# 16. PERFORM LEFT JOIN
# =============================================================================


joined = sample.merge(
    reference_for_join,
    on="_JOIN_KEY",
    how="left",
    indicator=True,
    validate="many_to_one"
)


# =============================================================================
# 17. IDENTIFY MATCHED AND UNMATCHED SAMPLES
# =============================================================================


matched_mask = (
    joined["_merge"]
    == "both"
)


unmatched_mask = (
    joined["_merge"]
    == "left_only"
)


# =============================================================================
# 18. ORIGINAL SAMPLE COLUMNS FOR UNMATCHED LOG
# =============================================================================


original_sample_columns = [
    column

    for column in sample.columns

    if column not in {
        "_JOIN_KEY",
        "_ORIGINAL_ORDER"
    }
]


# =============================================================================
# 19. CREATE UNMATCHED TABLE
# =============================================================================


unmatched = joined.loc[
    unmatched_mask,
    original_sample_columns
].copy()


# =============================================================================
# 20. CREATE MATCHED TABLE
# =============================================================================


matched = joined.loc[
    matched_mask
].copy()


# =============================================================================
# 21. RESTORE ORIGINAL SAMPLE ORDER
# =============================================================================


matched = matched.sort_values(
    "_ORIGINAL_ORDER"
).reset_index(
    drop=True
)


unmatched = unmatched.reset_index(
    drop=True
)


# =============================================================================
# 22. REMOVE INTERNAL COLUMNS
# =============================================================================


matched = matched.drop(
    columns=[
        "_JOIN_KEY",
        "_ORIGINAL_ORDER",
        "_merge"
    ],
    errors="ignore"
)


# =============================================================================
# 23. EXPORT MATCHED SAMPLE METADATA
# =============================================================================


matched.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 24. EXPORT UNMATCHED SAMPLE LOG
# =============================================================================


unmatched.to_csv(
    LOG_FILE,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 25. PROCESS SUMMARY
# =============================================================================


total_rows = len(
    sample
)


matched_rows = len(
    matched
)


unmatched_rows = len(
    unmatched
)


if (
    matched_rows
    +
    unmatched_rows
    != total_rows
):

    raise RuntimeError(
        "Internal validation failed: matched + unmatched "
        "does not equal the original number of samples."
    )


if total_rows > 0:

    match_percentage = (
        matched_rows
        / total_rows
        * 100
    )


else:

    match_percentage = 0


# =============================================================================
# 26. FINAL REPORT
# =============================================================================


print(
    "\n"
    +
    "=" * 60
)


print(
    "METADIV SAMPLE METADATA ASSIGNATION COMPLETED"
)


print(
    "=" * 60
)


print(
    f"\nOriginal samples:  "
    f"{total_rows:,}"
)


print(
    f"Matched samples:   "
    f"{matched_rows:,}"
)


print(
    f"Unmatched samples: "
    f"{unmatched_rows:,}"
)


print(
    f"Matching success:  "
    f"{match_percentage:.2f}%"
)


print(
    "\nOutput folder:"
)


print(
    OUTPUT_DIR
)


print(
    "\nMatched metadata:"
)


print(
    OUTPUT_FILE
)


print(
    "\nUnmatched samples:"
)


print(
    LOG_FILE
)


if DUPLICATE_LOG_FILE.exists():

    print(
        "\nDuplicated reference IDs:"
    )

    print(
        DUPLICATE_LOG_FILE
    )


# =============================================================================
# 27. DISPLAY RESULTS
# =============================================================================


print(
    "\nPreview of matched sample metadata:"
)


display(
    matched.head(20)
)


if unmatched_rows > 0:

    print(
        "\nPreview of unmatched samples:"
    )

    display(
        unmatched.head(20)
    )


else:

    print(
        "\nAll samples were successfully matched."
    )


MetaDiv sample_metadata_assignation

Project directory:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_metadata

Sample metadata:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_metadata\source_metadata\sample_metadata_to_fill.csv

Reference metadata:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_metadata\source_metadata\reference_metadata.csv

Columns used for sample matching:
  Sample metadata    -> sample_id
  Reference metadata -> search

No duplicated IDs were detected in the reference metadata.

Metadata columns to append: 56

METADIV SAMPLE METADATA ASSIGNATION COMPLETED

Original samples:  326
Matched samples:   73
Unmatched samples: 253
Matching success:  22.39%

Output folder:
C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\MetaDiv_Utilities\MetaDiv_metadata\merged_metadata

Matched metadata:
C:\Users\berna\Desktop\PAPER 

,sample_id,Group,Site,Treatment,Latitude,Longitude,sampleID,codigo_Atlas,codigo_fastq,codigo_int,...,C_P,N_P,Cmic,Nmic,Cmic_Nmic,CO2,BGLU,POX,LEU,FME
0,MXA_G5134,NaN,NaN,NaN,NaN,NaN,MXA_G5134,G5134,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
1,MXA_G5135,NaN,NaN,NaN,NaN,NaN,MXA_G5135,G5135,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
2,MXA_G5136,NaN,NaN,NaN,NaN,NaN,MXA_G5136,G5136,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
3,MXA_G5137,NaN,NaN,NaN,NaN,NaN,MXA_G5137,G5137,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
4,MXA_G5138,NaN,NaN,NaN,NaN,NaN,MXA_G5138,G5138,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
5,MXA_G5139,NaN,NaN,NaN,NaN,NaN,MXA_G5139,G5139,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
6,MXA_MX10,NaN,NaN,NaN,NaN,NaN,MXA_MX10,MX10,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
7,MXA_MX11,NaN,NaN,NaN,NaN,NaN,MXA_MX11,MX11,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
8,MXA_MX12,NaN,NaN,NaN,NaN,NaN,MXA_MX12,MX12,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_
9,MXA_MX13,NaN,NaN,NaN,NaN,NaN,MXA_MX13,MX13,NaN,NaN,...,_,_,_,_,_,_,_,_,_,_



Preview of unmatched samples:


,sample_id,Group,Site,Treatment,Latitude,Longitude
0,MXB_B01,NaN,NaN,NaN,NaN,NaN
1,MXB_B02,NaN,NaN,NaN,NaN,NaN
2,MXB_B02_L,NaN,NaN,NaN,NaN,NaN
3,MXB_B03,NaN,NaN,NaN,NaN,NaN
4,MXB_B03_L,NaN,NaN,NaN,NaN,NaN
5,MXB_B04,NaN,NaN,NaN,NaN,NaN
6,MXB_B04_L,NaN,NaN,NaN,NaN,NaN
7,MXB_B05,NaN,NaN,NaN,NaN,NaN
8,MXB_B05_L,NaN,NaN,NaN,NaN,NaN
9,MXB_B06,NaN,NaN,NaN,NaN,NaN
